In [1]:
import sys
!{sys.executable} -m pip install rdflib owlrl

# utilities
import pandas as pd

# libraries to handle triples and graphs
from rdflib import Graph, Namespace, URIRef, Literal, BNode
from rdflib.namespace import RDFS, RDF, OWL
from rdflib.collection import Collection


# libraries to handle reasoning
from owlrl import DeductiveClosure, OWLRL_Semantics

# Loading Graphs

In [2]:
# utility variable : list of namespaces that we will need for querying
namespaces = {'rdf': RDF, 'rdfs' : RDFS, "": 'http://example.org/', 'owl': OWL}

In [3]:
# initialise an empty graph
myKnowledgeGraph = Graph()

# parse a knowledge graph from a local file
myKnowledgeGraph = Graph().parse("./data/kitchen-exercise.ttl")

Let's see how many triples the graph contains

In [4]:
print("Graph has %s statements." % len(myKnowledgeGraph))

Graph has 211 statements.


Let's try adding and removing triples to see what changes in the graph.

In [5]:
# Adding triples
myKnowledgeGraph.add((URIRef("http://example.org/bed2"),RDF.type , URIRef("http://example.org/Bed")))

# print the length of the graph
print("My Kitchen graph has now %d triples." % len(myKnowledgeGraph))

#removing triples
myKnowledgeGraph.remove((URIRef("http://example.org/bed2"),RDF.type , URIRef("http://example.org/Bed")))

# print again the length of the graph
print("My Kitchen graph has now %d triples." % len(myKnowledgeGraph))

My Kitchen graph has now 212 triples.
My Kitchen graph has now 211 triples.


# Querying with RDFlib (without reasoning)

We can now query the graph we created. We start by asking which subjects are there in the graph.

In [6]:
# this is equivalent to saying : for i in myKnowledgeGraph.subjects() : print i
list(i for i in myKnowledgeGraph.subjects() if type(i) != BNode)

[rdflib.term.URIRef('http://example.org/Sofa'),
 rdflib.term.URIRef('http://example.org/bookshelf1'),
 rdflib.term.URIRef('http://example.org/bed1'),
 rdflib.term.URIRef('http://example.org/room3'),
 rdflib.term.URIRef('http://example.org/bookshelf1'),
 rdflib.term.URIRef('http://example.org/door3'),
 rdflib.term.URIRef('http://example.org/table1'),
 rdflib.term.URIRef('http://example.org/table3'),
 rdflib.term.URIRef('http://example.org/stove1'),
 rdflib.term.URIRef('http://example.org/contains'),
 rdflib.term.URIRef('http://example.org/door1'),
 rdflib.term.URIRef('http://example.org/door4'),
 rdflib.term.URIRef('http://example.org/Fridge'),
 rdflib.term.URIRef('http://example.org/bookshelf3'),
 rdflib.term.URIRef('http://example.org/Sink'),
 rdflib.term.URIRef('http://example.org/Kitchen'),
 rdflib.term.URIRef('http://example.org/Wardrobe'),
 rdflib.term.URIRef('http://example.org/Stove'),
 rdflib.term.URIRef('http://example.org/Chair'),
 rdflib.term.URIRef('http://example.org/room2

In [7]:
sorted(list(kitchen_class for kitchen_class in myKnowledgeGraph.subjects(RDF.type, OWL.Class) if type(kitchen_class) != BNode))

[rdflib.term.URIRef('http://example.org/Bed'),
 rdflib.term.URIRef('http://example.org/BedRoom'),
 rdflib.term.URIRef('http://example.org/Chair'),
 rdflib.term.URIRef('http://example.org/Door'),
 rdflib.term.URIRef('http://example.org/Fridge'),
 rdflib.term.URIRef('http://example.org/Kitchen'),
 rdflib.term.URIRef('http://example.org/KitchenObject'),
 rdflib.term.URIRef('http://example.org/LivingRoom'),
 rdflib.term.URIRef('http://example.org/Location'),
 rdflib.term.URIRef('http://example.org/Object'),
 rdflib.term.URIRef('http://example.org/Room'),
 rdflib.term.URIRef('http://example.org/Shelf'),
 rdflib.term.URIRef('http://example.org/Sink'),
 rdflib.term.URIRef('http://example.org/Sofa'),
 rdflib.term.URIRef('http://example.org/Stove'),
 rdflib.term.URIRef('http://example.org/Table'),
 rdflib.term.URIRef('http://example.org/Wardrobe'),
 rdflib.term.URIRef('https://example.org/Box')]

Let's ask whether 'door2' is closed.



In [8]:
openDoor2 = myKnowledgeGraph.value(URIRef('http://example.org/door3'), URIRef("http://example.org/hasStatus"))

if openDoor2 == "closed":
  print(True)
else : print(False)

False


We can update the status of a door and ask again if there is any door open.

In [9]:
# NB: you need to remove the old triple from the graph, and then add a new one. Else they will both be stored in the KG!
myKnowledgeGraph.remove((URIRef("http://example.org/door2"),URIRef('http://example.org/hasStatus') , Literal("open")))
myKnowledgeGraph.add((URIRef("http://example.org/door2"),URIRef('http://example.org/hasStatus') , Literal("closed")))


# ask which doors are closed
for s,p,o in myKnowledgeGraph.triples((None, URIRef("http://example.org/hasStatus"), None )):
  if o == Literal("closed") : print(s)

http://example.org/door2


# Reasoning and querying

We can now start a reasoner and generate triples we did not know before.

In [10]:
# initialise new graphs : asserted (triples stated in the ttl file), inferred (triples generated by the reasoner)
asserted = Graph()
inferred = Graph()

asserted = asserted.parse("./data/kitchen-exercise.ttl")
DeductiveClosure(OWLRL_Semantics).expand(myKnowledgeGraph) # this function will run an OWL RL reasoner and expand the asserted KG with inferred triples
# NB : the .expand() function will add the inferred triples to the original asserted graph. Thankfully, we saved a copy of the original asserted KG into the variable 'asserted'!

# we can now check which are the inferred triples by substracting the expanded and original graph.
inferred = myKnowledgeGraph - asserted

Let's now check how many triples these graphs have.

In [11]:
print("asserted {}, inferred {}, total {}".format(len(asserted), len(inferred),len(myKnowledgeGraph)))


asserted 211, inferred 589, total 737


Based on the restriction

```
:Kitchen owl:equivalentClass [ rdf:type owl:Restriction ;
                               owl:onProperty :contains ;
                               owl:someValuesFrom ( :Fridge :Sink :Stove )
                              ]
```
we know that the class `Kitchen` should contain at least an element from the class `Fridge`, `Sink` or `Stove`.

From the asserted triples, we know that `:room2` contains `:fridge1`, `:stove1` and `:sink1` .

If the reasoner worked, we can ask the inferred graph whether `:room2` is a kitchen.

In [12]:
EX = Namespace("http://example.org/")

# NB : EX['room2'] is now equivalent to URIRef("http://example.org/room2")
for s,p,o in inferred.triples((EX['room2'],RDF.type,None)):
      print(s,p,o)

http://example.org/room2 http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2002/07/owl#Thing
http://example.org/room2 http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://example.org/Location
http://example.org/room2 http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://example.org/Kitchen
http://example.org/room2 http://www.w3.org/1999/02/22-rdf-syntax-ns#type n7156b4662da94450bec09aa0a2651420b10


Now compare with what the asserted graph knows about `:room2`.   

In [13]:
# NB : EX['room2'] is now equivalent to URIRef("http://example.org/room2")
for s,p,o in asserted.triples((EX['room2'],RDF.type,None)):
      print(s,p,o)

http://example.org/room2 http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2002/07/owl#NamedIndividual
http://example.org/room2 http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://example.org/Room


The reasoner also reasons about object property contraints.

For instance, we know that the property `:contains` is inverse of `:hasLocation`.

In [14]:
# select objects from room4
for s,p,o in asserted.triples( (EX['room4'],EX['contains'],None)  ):
  print (o)

# Try replacing the triple pattern above with the triple (and remember to print the subject instead of the object)

Nothing is known about `:room4` in the asserted graph. Hence the result is empty.

Go check yourself in the `.ttl` file!



In [15]:
# compare with what is known about room4 in the inferred graph
for s,p,o in inferred.triples( (EX['room4'],EX['contains'],None)  ):
  print (o)

# Now we have the right inference!

http://example.org/table2
http://example.org/box3
http://example.org/table1
http://example.org/bookshelf3
http://example.org/box2


# Exercises

**Exercise 1**

Extend the taxonomy of Kitchen classes, with e.g. movable and non-movable classes

- Serialize the taxonomy into a .ttl file and parse it again into a new graph.

- Query the new graph and check if your taxonomy was taken into account.




In [16]:
# your code here

EX = Namespace("http://example.org/")

taxG = Graph()
taxG.bind("", EX)
taxG.bind("rdf", RDF)
taxG.bind("rdfs", RDFS)
taxG.bind("owl", OWL)

taxG.add((EX.MovableObject, RDF.type, OWL.Class))
taxG.add((EX.NonMovableObject, RDF.type, OWL.Class))
taxG.add((EX.MovableObject, OWL.disjointWith, EX.NonMovableObject))

for cls in [EX.Bed, EX.Sofa, EX.Bookshelf, EX.Fridge, EX.Sink, EX.Stove, EX.Door, EX.Wall, EX.Room]:
    taxG.add((cls, RDF.type, OWL.Class))

for movable_cls in [EX.Bed, EX.Sofa, EX.Bookshelf]:
    taxG.add((movable_cls, RDFS.subClassOf, EX.MovableObject))

for nonmovable_cls in [EX.Fridge, EX.Sink, EX.Stove, EX.Door, EX.Wall, EX.Room]:
    taxG.add((nonmovable_cls, RDFS.subClassOf, EX.NonMovableObject))

out_path = "./data/taxonomy-ex1.ttl"
taxG.serialize(destination=out_path, format="turtle")
print(f"saved taxonomy to: {out_path} with {len(taxG)} triples")

taxG2 = Graph().parse(out_path, format="turtle")
taxG2.bind("", EX)

print("parsed taxonomy graph has %s statements." % len(taxG2))

print("\nsubclasses of movableobject:")
for s in sorted(set(taxG2.subjects(RDFS.subClassOf, EX.MovableObject))):
    if type(s) != BNode:
        print(" -", s)

print("\nsubclasses of nonmovableobject:")
for s in sorted(set(taxG2.subjects(RDFS.subClassOf, EX.NonMovableObject))):
    if type(s) != BNode:
        print(" -", s)

saved taxonomy to: ./data/taxonomy-ex1.ttl with 21 triples
parsed taxonomy graph has 21 statements.

subclasses of movableobject:
 - http://example.org/Bed
 - http://example.org/Bookshelf
 - http://example.org/Sofa

subclasses of nonmovableobject:
 - http://example.org/Door
 - http://example.org/Fridge
 - http://example.org/Room
 - http://example.org/Sink
 - http://example.org/Stove
 - http://example.org/Wall


**Exercise 2**

Build a restriction for the living room or bedroom, similar to the one we have for the kitchen
(i.e. a bedroom is a room that contains at least one bed, a livingroom has at least a sofa and a bookshelf)

- Create a new file .ttl and parse it in a new graph.

- Run the reasoner and check if your restriction was taken into account.

- _harder variant_ : try using ```owl:minCardinality``` instead of ```owl:someValuesFrom``` ! Tip : you may need to create a new object property.



In [17]:
EX = Namespace("http://example.org/")

asserted = Graph().parse("./data/kitchen-exercise.ttl")
asserted.bind("", EX)
asserted.bind("owl", OWL)
asserted.bind("rdf", RDF)
asserted.bind("rdfs", RDFS)

ax = Graph()
ax.bind("", EX)
ax.bind("owl", OWL)
ax.bind("rdf", RDF)
ax.bind("rdfs", RDFS)

for cls in [EX.Bedroom, EX.LivingRoom, EX.Room, EX.Bed, EX.Sofa, EX.Bookshelf]:
    ax.add((cls, RDF.type, OWL.Class))

ax.add((EX.contains, RDF.type, OWL.ObjectProperty))

bedroom_restr = BNode()
ax.add((bedroom_restr, RDF.type, OWL.Restriction))
ax.add((bedroom_restr, OWL.onProperty, EX.contains))
ax.add((bedroom_restr, OWL.someValuesFrom, EX.Bed))
ax.add((EX.Bedroom, OWL.equivalentClass, bedroom_restr))

sofa_restr = BNode()
ax.add((sofa_restr, RDF.type, OWL.Restriction))
ax.add((sofa_restr, OWL.onProperty, EX.contains))
ax.add((sofa_restr, OWL.someValuesFrom, EX.Sofa))

bookshelf_restr = BNode()
ax.add((bookshelf_restr, RDF.type, OWL.Restriction))
ax.add((bookshelf_restr, OWL.onProperty, EX.contains))
ax.add((bookshelf_restr, OWL.someValuesFrom, EX.Bookshelf))

intersection = BNode()
ax.add((intersection, RDF.type, OWL.Class))
list_node = BNode()
Collection(ax, list_node, [sofa_restr, bookshelf_restr])
ax.add((intersection, OWL.intersectionOf, list_node))
ax.add((EX.LivingRoom, OWL.equivalentClass, intersection))

ax_path = "./data/room-restrictions-ex2.ttl"
ax.serialize(destination=ax_path, format="turtle")
print(f"saved new restrictions to: {ax_path} with {len(ax)} triples")

ax2 = Graph().parse(ax_path, format="turtle")
ax2.bind("", EX)

combined = Graph()
for t in asserted:
    combined.add(t)
for t in ax2:
    combined.add(t)

combined_before = Graph()
for t in combined:
    combined_before.add(t)

DeductiveClosure(OWLRL_Semantics).expand(combined)
inferred = combined - combined_before

print("asserted+axioms {}, inferred {}, total {}".format(len(combined_before), len(inferred), len(combined)))

beds = set(asserted.subjects(RDF.type, EX.Bed))
bedroom_candidates = set()
for bed in beds:
    for room in asserted.subjects(EX.contains, bed):
        bedroom_candidates.add(room)

sofas = set(asserted.subjects(RDF.type, EX.Sofa))
books = set(asserted.subjects(RDF.type, EX.Bookshelf))

rooms_with_sofa = set()
for sofa in sofas:
    rooms_with_sofa |= set(asserted.subjects(EX.contains, sofa))

rooms_with_books = set()
for b in books:
    rooms_with_books |= set(asserted.subjects(EX.contains, b))

livingroom_candidates = rooms_with_sofa & rooms_with_books

print("\nbedroom candidates from asserted KG:", sorted(map(str, bedroom_candidates)))
print("livingRoom candidates from asserted KG:", sorted(map(str, livingroom_candidates)))

def print_inferred_types(resource_uriref):
    types = sorted(set(o for o in inferred.objects(resource_uriref, RDF.type)))
    if types:
        print("\ninferred rdf:types for", resource_uriref)
        for t in types:
            print(" -", t)
    else:
        print("\nno inferred rdf:types for", resource_uriref)

for r in sorted(bedroom_candidates, key=str):
    print_inferred_types(r)

for r in sorted(livingroom_candidates, key=str):
    print_inferred_types(r)

saved new restrictions to: ./data/room-restrictions-ex2.ttl with 24 triples
asserted+axioms 230, inferred 570, total 800

bedroom candidates from asserted KG: []
livingRoom candidates from asserted KG: []


**Exercise 3**

Update the status of a door (from e.g. open to close) or the coordinates of an object.

- Use the functions
```
graph.remove((UriRef(#subject),UriRef(#predicate ),UriRef( #object )))
graph.add((UriRef(#subject),UriRef(#predicate ),UriRef( #object )) )
```
to remove the old triple and add a new one.



- Save the graph using the function
```graph.serialise()```

- check with your favourite text editor if your KG has been updated!

In [18]:
# your code here
EX = Namespace("http://example.org/")

g = Graph().parse("./data/kitchen-exercise.ttl")
g.bind("", EX)

door = EX.door2
hasStatus = EX.hasStatus

old_status = g.value(door, hasStatus)
print("old status of door2:", old_status)

for o in list(g.objects(door, hasStatus)):
    g.remove((door, hasStatus, o))

g.add((door, hasStatus, Literal("closed")))

new_status = g.value(door, hasStatus)
print("new status of door2:", new_status)

out_path = "./data/kitchen-exercise-updated-ex3.ttl"
g.serialize(destination=out_path, format="turtle")
print(f"saved updated KG to: {out_path} (triples: {len(g)})")

print("\ndoors with status 'closed':")
for s, p, o in g.triples((None, hasStatus, Literal("closed"))):
    print(" -", s)

old status of door2: open
new status of door2: closed
saved updated KG to: ./data/kitchen-exercise-updated-ex3.ttl (triples: 211)

doors with status 'closed':
 - http://example.org/door2
